## Data Preparation

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# !pip install torch transformers pytorch-crf tqdm pandas matplotlib seaborn

In [3]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoConfig

In [ ]:
class NERDataset(Dataset):
    def __init__(self, data_path, tokenizer, label_pad_id=-100, max_length=128):
        with open(data_path, "r", encoding="utf-8") as f:
            raw = json.load(f)["examples"]
        self.data = raw
        self.tokenizer = tokenizer
        self.label_pad_id = label_pad_id
        self.max_length = max_length

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        tokens = self.data[idx]["tokens"]
        ner_tags = self.data[idx]["ner_tags"]
 
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors="pt"
        )

        word_ids = encoding.word_ids(batch_index=0)
        aligned_labels = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(self.label_pad_id)
            elif word_idx != previous_word_idx:
                aligned_labels.append(ner_tags[word_idx])
            else:
                aligned_labels.append(self.label_pad_id)
            previous_word_idx = word_idx
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(aligned_labels, dtype=torch.long)
        }

        return item

In [ ]:
def load_label_info(model_name):
    config = AutoConfig.from_pretrained(model_name)
    id2label = config.id2label
    label2id = config.label2id
    num_labels = config.num_labels

    label_info = {
        "id2label": id2label,
        "label2id": label2id,
        "num_labels": num_labels
    }

    return label_info

def create_dataloaders(
        train_path, val_path, test_path,
        model_name,
        batch_size=32,
        max_length=128
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_dataset = NERDataset(train_path, tokenizer, max_length=max_length)
    val_dataset = NERDataset(val_path, tokenizer, max_length=max_length)
    test_dataset = NERDataset(test_path, tokenizer, max_length=max_length)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

## Model Architecture

In [ ]:
from torch import nn
from torchcrf import CRF
from transformers import AutoModel, AutoConfig

In [ ]:
class CRFOutputLayer(nn.Module):
    def __init__(self, hidden_dim, num_labels):
        super().__init__()
        self.fc = nn.Linear(hidden_dim, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, outputs, labels=None, mask=None):
        emissions = self.fc(outputs)

        if labels is not None:
            if mask is None:
                mask = torch.ones_like(labels, dtype=torch.bool)
            else:
                mask = mask.bool()
            
            mask[:, 0] = True
            
            labels_crf = labels.clone()
            labels_crf[labels == -100] = 0
            
            log_likelihood = self.crf(emissions, tags=labels_crf, mask=mask, reduction="token_mean")
            loss = -log_likelihood
            return {"logits": emissions, "loss": loss}
        else:
            if mask is None:
                mask = torch.ones(outputs.shape[:2], dtype=torch.bool, device=outputs.device)
            pred = self.crf.decode(emissions, mask=mask.bool())
            return {"logits": emissions, "pred": pred}


In [ ]:
class BaseNERModel(nn.Module):
    def __init__(self, num_labels, use_crf=False):
        super().__init__()
        self.num_labels = num_labels
        self.use_crf = use_crf

    def forward(self, input_ids, attention_mask, labels=None):
        raise NotImplementedError("Forward method must be implemented in subclass.")

In [ ]:
class QueryNERTeacher(BaseNERModel):
    def __init__(self, model_name, label_info, use_crf=False):
        super().__init__(num_labels=label_info["num_labels"], use_crf=use_crf)

        self.config = AutoConfig.from_pretrained(
            model_name,
            num_labels=label_info["num_labels"],
            id2label=label_info["id2label"],
            label2id=label_info["label2id"]
        )

        self.bert = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(0.3)

        if self.use_crf:
            self.crf_output = CRFOutputLayer(self.config.hidden_size, self.config.num_labels)
        else:
            self.classifier = nn.Linear(self.config.hidden_size, self.config.num_labels)
            self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_ids, attention_mask, labels=None):

        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)

        if self.use_crf:
            mask = attention_mask.bool()
            result = self.crf_output(sequence_output, labels=labels, mask=mask)
            return result

        else:
            logits = self.classifier(sequence_output)
            if labels is not None:
                loss = self.loss_fn(logits.view(-1, self.num_labels), labels.view(-1))
                return {"logits": logits, "loss": loss}
            else:
                pred = logits.argmax(dim=-1)
                return {"logits": logits, "pred": pred}

In [ ]:
class DistilBERTStudent(BaseNERModel):
    def __init__(self, model_name="distilbert-base-uncased", label_info=None, use_crf=False):
        self.use_crf = use_crf
        self.num_labels = label_info["num_labels"]
        super().__init__(num_labels=self.num_labels, use_crf=self.use_crf)

        self.config = AutoConfig.from_pretrained(
            model_name,
            num_labels=label_info["num_labels"],
            id2label=label_info["id2label"],
            label2id=label_info["label2id"]
        )

        self.bert = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(0.3)

        if self.use_crf:
            self.crf_output = CRFOutputLayer(self.config.hidden_size, self.num_labels)
        else:
            self.classifier = nn.Linear(self.config.hidden_size, self.num_labels)
            self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)

        if self.use_crf:
            mask = attention_mask.bool()
            result = self.crf_output(sequence_output, labels=labels, mask=mask)
            return result
        else:
            logits = self.classifier(sequence_output)
            if labels is not None:
                loss = self.loss_fn(logits.view(-1, self.num_labels), labels.view(-1))
                return {"logits": logits, "loss": loss}
            else:
                pred = logits.argmax(dim=-1)
                return {"logits": logits, "pred": pred}


In [ ]:
class TinyBertStudent(BaseNERModel):
    def __init__(self, model_name="huawei-noah/TinyBERT_General_4L_312D", label_info=None, use_crf=False):
        self.use_crf = use_crf
        self.num_labels = label_info["num_labels"]
        super().__init__(num_labels=self.num_labels, use_crf=self.use_crf)

        self.config = AutoConfig.from_pretrained(
            model_name,
            num_labels=label_info["num_labels"],
            id2label=label_info["id2label"],
            label2id=label_info["label2id"]
        )

        self.bert = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(0.3)

        if self.use_crf:
            self.crf_output = CRFOutputLayer(self.config.hidden_size, self.num_labels)
        else:
            self.classifier = nn.Linear(self.config.hidden_size, self.num_labels)
            self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)

        if self.use_crf:
            mask = attention_mask.bool()
            result = self.crf_output(sequence_output, labels=labels, mask=mask)
            return result
        else:
            logits = self.classifier(sequence_output)
            if labels is not None:
                loss = self.loss_fn(logits.view(-1, self.num_labels), labels.view(-1))
                return {"logits": logits, "loss": loss}
            else:
                pred = logits.argmax(dim=-1)
                return {"logits": logits, "pred": pred}

In [ ]:
class BiLSTMStudent(BaseNERModel):
    def __init__(
            self,  
            use_crf=False,
            model_name_for_vocab = 'bert-base-uncased',
            emb_dim = 300,
            lstm_hidden = 300,
            label_info = None,
            pad_token_id = 0,
            teacher_model = None
        ):
        self.use_crf = use_crf
        self.num_labels = label_info["num_labels"]
        super().__init__(self.num_labels, use_crf)

        config = AutoConfig.from_pretrained(model_name_for_vocab)
        vocab_size = config.vocab_size
        pad_token_id = config.pad_token_id

        if teacher_model is not None:
            self.embedding = teacher_model.bert.embeddings.word_embeddings
            for p in self.embedding.parameters():
                p.requires_grad = False
            emb_dim_eff = self.embedding.embedding_dim
        else:
            emb_dim_eff = emb_dim
            self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_token_id)
        self.dropout = nn.Dropout(0.1)
        self.lstm = nn.LSTM(
            input_size=emb_dim_eff,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(lstm_hidden * 2, self.num_labels)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

        if self.use_crf:
            self.crf_output = CRFOutputLayer(hidden_dim=lstm_hidden * 2, num_labels=self.num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        emb = self.embedding(input_ids)
        emb = self.dropout(emb)
        outputs, _ = self.lstm(emb)
        sequence_output = outputs

        if self.use_crf:
            mask = attention_mask.bool()
            result = self.crf_output(sequence_output, labels=labels, mask=mask)
            return result
        else:
            logits = self.classifier(sequence_output)
            if labels is not None:
                loss = self.loss_fn(logits.view(-1, self.num_labels), labels.view(-1))
                return {"logits": logits, "loss": loss}
            else:
                pred = logits.argmax(dim=-1)
                return {"logits": logits, "pred": pred}


## Knowledge Distillation Scheme

In [4]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

In [ ]:
def kl_divergence_loss_masked(student_logits, teacher_logits, temperature, mask=None, eps=1e-12):
    """
    Compute masked KL divergence loss for knowledge distillation.
    
    Args:
        student_logits: (B, L, C) logits from student
        teacher_logits: (B, L, C) logits from teacher
        temperature: Temperature for softening distributions
        mask: (B, L) attention mask (1 for valid tokens)
        eps: Small constant for numerical stability
    
    Returns:
        Scalar loss value
    """
    T = float(temperature)

    student_log_prob = F.log_softmax(student_logits / T, dim=-1)   # (B, L, C) > (Batch, SeqLen, NumClasses) > (8, 128, 37)
    teacher_prob = F.softmax(teacher_logits / T, dim=-1)           # (B, L, C)

    kl_elem = F.kl_div(student_log_prob, teacher_prob, reduction='none')  # (B, L, C)
    kl_token = kl_elem.sum(dim=-1)  # (B, L)

    if mask is not None:
        mask = mask.bool()
        valid_sum = mask.float().sum()
        if valid_sum.item() == 0:
            return torch.tensor(0.0, device=student_logits.device)
        kl_sum = (kl_token * mask.float()).sum()
        return (kl_sum / valid_sum) * (T * T)
    else:
        return kl_token.mean() * (T * T)

In [ ]:
def kl_divergence_loss_masked(student_logits, teacher_logits, temperature, mask=None, eps=1e-12):
    """
    Compute masked KL divergence loss for knowledge distillation.
    
    Args:
        student_logits: (B, L, C) logits from student
        teacher_logits: (B, L, C) logits from teacher
        temperature: Temperature for softening distributions
        mask: (B, L) attention mask (1 for valid tokens)
        eps: Small constant for numerical stability
    
    Returns:
        Scalar loss value
    """
    T = float(temperature)

    student_log_prob = F.log_softmax(student_logits / T, dim=-1)   # (B, L, C)
    teacher_prob = F.softmax(teacher_logits / T, dim=-1)           # (B, L, C)

    kl_elem = F.kl_div(student_log_prob, teacher_prob, reduction='none')  # (B, L, C)
    kl_token = kl_elem.sum(dim=-1)  # (B, L)

    if mask is not None:
        mask = mask.bool()
        valid_sum = mask.float().sum()
        if valid_sum.item() == 0:
            return torch.tensor(0.0, device=student_logits.device)
        kl_sum = (kl_token * mask.float()).sum()
        return (kl_sum / valid_sum) * (T * T)
    else:
        return kl_token.mean() * (T * T)


def _to_tensor_preds(preds, batch_size, seq_len, device):
    """
    Convert CRF decode output (list[list[int]] or list of tensors) into a tensor
    of shape (batch_size, seq_len) padded with 0s. Caller must mask invalid tokens.
    """
    pred_tensor = torch.zeros((batch_size, seq_len), dtype=torch.long, device=device)
    for i, p in enumerate(preds):
        if isinstance(p, torch.Tensor):
            p = p.tolist()
        L = len(p)
        if L > 0:
            pred_tensor[i, :L] = torch.tensor(p, dtype=torch.long, device=device)
    return pred_tensor


def _safe_get_pred_tensor(output, batch_size, seq_len, device):
    """
    Return a (batch, seq_len) tensor of predictions from model output.
    Handles:
      - output["pred"] is a tensor (batch, seq_len)
      - output["pred"] is a list of lists (per-seq predicted label ids)
      - output has no "pred" (use logits.argmax)
    """
    if "pred" in output:
        pred = output["pred"]
        if isinstance(pred, torch.Tensor):
            return pred.to(device)
        else:
            # assume list of lists
            return _to_tensor_preds(pred, batch_size, seq_len, device)
    elif "logits" in output:
        return output["logits"].argmax(dim=-1).to(device)
    else:
        raise ValueError("No 'pred' or 'logits' in model output to produce predictions.")


def _accumulate_confusion_counts(preds_flat, labels_flat):
    """
    Compute per-class TP, predicted_counts, actual_counts using vectors.
    preds_flat and labels_flat are 1D torch.Long tensors on CPU or device.
    Returns (tp_sum, pred_sum, actual_sum) and also total_tp, total_pred, total_actual per class sums.
    """
    if preds_flat.numel() == 0:
        return 0, 0, 0, None

    max_label = int(max(int(preds_flat.max().item()), int(labels_flat.max().item())))
    num_classes = max_label + 1

    tp_per_class = torch.zeros(num_classes, dtype=torch.long, device=preds_flat.device)
    pred_per_class = torch.zeros(num_classes, dtype=torch.long, device=preds_flat.device)
    actual_per_class = torch.zeros(num_classes, dtype=torch.long, device=preds_flat.device)

    for c in range(num_classes):
        pred_mask = preds_flat == c
        lab_mask = labels_flat == c
        tp_per_class[c] = int((pred_mask & lab_mask).sum().item())
        pred_per_class[c] = int(pred_mask.sum().item())
        actual_per_class[c] = int(lab_mask.sum().item())

    tp_sum = int(tp_per_class.sum().item())
    pred_sum = int(pred_per_class.sum().item())
    actual_sum = int(actual_per_class.sum().item())

    return tp_sum, pred_sum, actual_sum, (tp_per_class.cpu().numpy(), pred_per_class.cpu().numpy(), actual_per_class.cpu().numpy())


def _batch_metrics(pred_tensor, label_tensor, attention_mask):
    """
    pred_tensor: (B, L)
    label_tensor: (B, L) with -100 for ignored positions
    attention_mask: (B, L) with 1 for valid tokens
    Returns TP, predicted_count, actual_count (ints)
    """
    mask = attention_mask.bool()
    # also ensure labels not equal to -100 in valid positions
    valid = mask & (label_tensor != -100)
    if valid.sum().item() == 0:
        return 0, 0, 0

    preds_flat = pred_tensor[valid].view(-1)
    labels_flat = label_tensor[valid].view(-1)

    tp_sum, pred_sum, actual_sum, _ = _accumulate_confusion_counts(preds_flat, labels_flat)
    return tp_sum, pred_sum, actual_sum


def _final_metrics(tp_sum, pred_sum, actual_sum):
    """
    Compute micro precision, recall, f1 from aggregated counts.
    """
    precision = tp_sum / pred_sum if pred_sum > 0 else 0.0
    recall = tp_sum / actual_sum if actual_sum > 0 else 0.0
    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0.0
    return precision, recall, f1


class KDTrainer:
    """
    Trainer for both baseline (fine-tuning) and knowledge distillation.
    
    For baseline:
        teacher_model=None, alpha=0, beta=1
    
    For KD:
        teacher_model=<trained_model>, alpha=0.5, beta=0.5
    """
    
    def __init__(
        self,
        teacher_model,
        student_model,
        train_loader,
        val_loader,
        optimizer,
        scheduler=None,
        device="cuda",
        alpha=0.5,
        beta=0.5,
        temperature=2.0,
        scheduler_type="plateau"  # "plateau", "cosine", "step", or None
    ):
        """
        Args:
            teacher_model: Teacher model (None for baseline)
            student_model: Student model to train
            train_loader: Training DataLoader
            val_loader: Validation DataLoader
            optimizer: Optimizer for student model
            scheduler: Learning rate scheduler (optional)
            device: Device to use
            alpha: Weight for KD loss (0 for baseline)
            beta: Weight for student loss (1 for baseline)
            temperature: Temperature for KD
            scheduler_type: Type of scheduler for proper step() call
        """
        self.student = student_model.to(device)
        self.teacher = None  # ← FIX: Initialize to None
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.scheduler_type = scheduler_type
        self.device = device
        self.alpha = alpha
        self.beta = beta
        self.T = temperature

        # Setup teacher if provided
        if teacher_model is not None:
            self.teacher = teacher_model.to(device)
            self.teacher.eval()  # Set to eval mode
            for p in self.teacher.parameters():
                p.requires_grad = False
        
        # Validate configuration
        if self.alpha > 0 and self.teacher is None:
            raise ValueError("alpha > 0 requires a teacher model!")
        
        # Print training mode
        mode = "BASELINE" if self.teacher is None else "KNOWLEDGE DISTILLATION"
        print(f"\n{'='*60}")
        print(f"Training Mode: {mode}")
        print(f"Alpha (KD loss weight): {self.alpha}")
        print(f"Beta (Student loss weight): {self.beta}")
        print(f"Temperature: {self.T}")
        print(f"{'='*60}\n")

    def compute_losses(self, batch):
        """
        Compute losses for one batch.
        
        Returns:
            loss_total: Combined loss
            loss_kd: KD loss (0 if no teacher)
            loss_student: Student task loss
            pred_tensor: Predictions for metrics
            labels: Ground truth labels
            attention_mask: Attention mask
        """
        input_ids = batch["input_ids"].to(self.device)
        attention_mask = batch["attention_mask"].to(self.device)
        labels = batch["labels"].to(self.device)

        batch_size, seq_len = input_ids.shape

        # Get teacher logits if needed
        teacher_logits = None
        if self.alpha > 0 and self.teacher is not None:
            with torch.no_grad():
                self.teacher.eval()
                teacher_out = self.teacher(
                    input_ids=input_ids, 
                    attention_mask=attention_mask
                )
                teacher_logits = teacher_out["logits"]

        # Get student outputs
        student_out = self.student(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            labels=labels
        )
        student_logits = student_out["logits"]

        # Compute KD loss if teacher provided
        if teacher_logits is not None:
            loss_kd = kl_divergence_loss_masked(
                student_logits, 
                teacher_logits, 
                self.T, 
                mask=attention_mask
            )
        else:
            loss_kd = torch.tensor(0.0, device=self.device)

        # Get student task loss
        loss_student = student_out.get("loss", torch.tensor(0.0, device=self.device))

        # Combined loss
        loss_total = self.alpha * loss_kd + self.beta * loss_student

        # Get predictions for metrics
        pred_tensor = _safe_get_pred_tensor(student_out, batch_size, seq_len, self.device)

        return loss_total, loss_kd, loss_student, pred_tensor, labels, attention_mask

    def train_epoch(self):
        """Train for one epoch."""
        self.student.train()
        total_loss, total_kd, total_stu = 0.0, 0.0, 0.0
        tp_acc, pred_acc, actual_acc = 0, 0, 0

        for batch in tqdm(self.train_loader, desc="Training"):
            self.optimizer.zero_grad()
            
            loss_total, loss_kd, loss_student, pred_tensor, labels, attention_mask = \
                self.compute_losses(batch)
            
            loss_total.backward()
            self.optimizer.step()

            total_loss += float(loss_total.item())
            total_kd += float(loss_kd.item())
            total_stu += float(loss_student.item()) if isinstance(loss_student, torch.Tensor) else float(loss_student)

            tp, pred_count, actual_count = _batch_metrics(pred_tensor, labels, attention_mask)
            tp_acc += tp
            pred_acc += pred_count
            actual_acc += actual_count

        avg_loss = total_loss / len(self.train_loader)
        avg_kd = total_kd / len(self.train_loader)
        avg_stu = total_stu / len(self.train_loader)

        # Update scheduler if provided
        if self.scheduler:
            if self.scheduler_type == "plateau":
                self.scheduler.step(avg_loss)
            else:
                # For cosine, step, etc. that don't need metrics
                self.scheduler.step()

        precision, recall, f1 = _final_metrics(tp_acc, pred_acc, actual_acc)
        return avg_loss, avg_kd, avg_stu, precision, recall, f1

    def validate(self):
        """Validate on validation set."""
        self.student.eval()
        total_loss, total_kd, total_stu = 0.0, 0.0, 0.0
        tp_acc, pred_acc, actual_acc = 0, 0, 0

        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Validation"):
                loss_total, loss_kd, loss_student, pred_tensor, labels, attention_mask = \
                    self.compute_losses(batch)
                
                total_loss += float(loss_total.item())
                total_kd += float(loss_kd.item())
                total_stu += float(loss_student.item()) if isinstance(loss_student, torch.Tensor) else float(loss_student)

                tp, pred_count, actual_count = _batch_metrics(pred_tensor, labels, attention_mask)
                tp_acc += tp
                pred_acc += pred_count
                actual_acc += actual_count

        avg_loss = total_loss / len(self.val_loader)
        avg_kd = total_kd / len(self.val_loader)
        avg_stu = total_stu / len(self.val_loader)

        precision, recall, f1 = _final_metrics(tp_acc, pred_acc, actual_acc)
        return avg_loss, avg_kd, avg_stu, precision, recall, f1

    def train(self, num_epochs):
        """
        Train for multiple epochs.
        
        Args:
            num_epochs: Number of epochs to train
            
        Returns:
            history: Dictionary containing training history
        """
        history = {
            "train_loss": [], "val_loss": [],
            "train_kd": [], "val_kd": [],
            "train_stu": [], "val_stu": [],
            "train_precision": [], "train_recall": [], "train_f1": [],
            "val_precision": [], "val_recall": [], "val_f1": []
        }

        best_val_f1 = 0.0
        
        for epoch in range(1, num_epochs + 1):
            print(f"\n{'='*60}")
            print(f"EPOCH {epoch}/{num_epochs}")
            print(f"{'='*60}")
            
            train_loss, train_kd, train_stu, train_prec, train_rec, train_f1 = self.train_epoch()
            val_loss, val_kd, val_stu, val_prec, val_rec, val_f1 = self.validate()

            print(f"\nTrain Loss: {train_loss:.4f} (KD: {train_kd:.4f}, Student: {train_stu:.4f})")
            print(f"Val Loss:   {val_loss:.4f} (KD: {val_kd:.4f}, Student: {val_stu:.4f})")
            print(f"\nTrain Metrics -> P: {train_prec:.4f}, R: {train_rec:.4f}, F1: {train_f1:.4f}")
            print(f"Val Metrics   -> P: {val_prec:.4f}, R: {val_rec:.4f}, F1: {val_f1:.4f}")
            
            # Track best validation F1
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                print(f"✓ New best Val F1: {best_val_f1:.4f}")

            # Store history
            history["train_loss"].append(train_loss)
            history["train_kd"].append(train_kd)
            history["train_stu"].append(train_stu)
            history["val_loss"].append(val_loss)
            history["val_kd"].append(val_kd)
            history["val_stu"].append(val_stu)

            history["train_precision"].append(train_prec)
            history["train_recall"].append(train_rec)
            history["train_f1"].append(train_f1)
            history["val_precision"].append(val_prec)
            history["val_recall"].append(val_rec)
            history["val_f1"].append(val_f1)

        print(f"\n{'='*60}")
        print(f"Training Complete!")
        print(f"Best Val F1: {best_val_f1:.4f}")
        print(f"{'='*60}\n")

        return history

## Distillation

In [ ]:
"""
Knowledge Distillation Training Script for NER Models

Configuration:
- 2 Teachers (CRF and NoCRF)
- 3 Students (DistilBERT, TinyBERT, BiLSTM)
- 3 CRF Combinations (both CRF, teacher CRF only, both NoCRF)
- 2 Alpha/Beta pairs ((0.5,0.5), (0.7,0.3))
- 2 Temperatures (2, 4)

Total: 2 × 3 × 3 × 2 × 2 = 72 experiments
"""

import json
import os
import torch
import torch.optim as optim
from datetime import datetime
from pathlib import Path
 
# Import your existing modules
# from your_module import (
#     NERDataset, create_dataloaders, load_label_info,
#     QueryNERTeacher, DistilBERTStudent, TinyBertStudent, BiLSTMStudent,
#     KDTrainer
# )


class EarlyStopping:
    """Early stopping to stop training when validation metric stops improving."""
    
    def __init__(self, patience=3, min_delta=0.001, mode='max'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0
        
    def __call__(self, val_metric, epoch):
        score = val_metric
        
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
            return False
        
        if self.mode == 'max':
            improved = score > self.best_score + self.min_delta
        else:
            improved = score < self.best_score - self.min_delta
        
        if improved:
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
        else:
            self.counter += 1
            print(f"  EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
                print(f"  ✓ Early stopping triggered! Best epoch was {self.best_epoch}")
        
        return self.early_stop


def create_kd_experiment_config():
    """Generate all 72 KD experiment configurations"""
    
    # Teacher configurations
    teachers = [
        {
            "name": "teacher_crf",
            "checkpoint": "/kaggle/input/teacher-processed-2e-5-nocrf-best/pytorch/default/3/teacher_processed_2e-5_crf_best_token_mean.pt",
            "model_class": "QueryNERTeacher",
            "model_path": "bert-base-uncased",
            "use_crf": True
        },
        {
            "name": "teacher_nocrf",
            "checkpoint": "/kaggle/input/teacher-processed-2e-5-nocrf-best/pytorch/default/3/teacher_processed_2e-5_nocrf_best_token_mean.pt",
            "model_class": "QueryNERTeacher",
            "model_path": "bert-base-uncased",
            "use_crf": False
        }
    ]
    
    # Student configurations
    students = [
        {
            "name": "distilbert",
            "model_class": "DistilBERTStudent",
            "model_path": "distilbert-base-uncased",
            "lr": 2e-5
        },
        {
            "name": "tinybert",
            "model_class": "TinyBertStudent",
            "model_path": "huawei-noah/TinyBERT_General_4L_312D",
            "lr": 2e-5
        },
        {
            "name": "bilstm",
            "model_class": "BiLSTMStudent",
            "model_path": "bert-base-uncased",  # For vocab
            "lr": 3e-3  # BiLSTM typically needs higher LR
        }
    ]
    
    # Dataset (using processed data)
    dataset = {
        "train": "/kaggle/input/queryner-dataset/data/processed/train.json",
        "val": "/kaggle/input/queryner-dataset/data/processed/validation.json",
        "test": "/kaggle/input/queryner-dataset/data/processed/test.json"
    }
    
    # CRF combinations
    # Format: (teacher_crf, student_crf, combination_name)
    crf_combinations = [
        (True, True, "both_crf"),      # Both use CRF
        (True, False, "teacher_crf"),   # Only teacher uses CRF
        (False, False, "both_nocrf")    # Neither uses CRF
    ]
    
    # Alpha/Beta pairs (alpha for KD loss, beta for student loss)
    alpha_beta_pairs = [
        (0.5, 0.5, "alpha0.5_beta0.5"),
        (0.7, 0.3, "alpha0.7_beta0.3")
    ]
    
    # Temperatures
    temperatures = [
        (2.0, "temp2"),
        (4.0, "temp4")
    ]
    
    # Generate all combinations
    experiments = []
    exp_id = 1
    
    for teacher in teachers:
        for student in students:
            if student["name"] != "bilstm":
                continue
            for teacher_crf_required, student_crf, crf_name in crf_combinations:
                # Skip invalid combinations
                # If combination requires teacher with CRF but teacher doesn't have it, skip
                if teacher_crf_required and not teacher["use_crf"]:
                    continue
                # If combination requires teacher without CRF but teacher has it, skip
                if not teacher_crf_required and teacher["use_crf"]:
                    continue
                
                for alpha, beta, ab_name in alpha_beta_pairs:
                    for temp, temp_name in temperatures:
                        exp = {
                            "id": exp_id,
                            "teacher_name": teacher["name"],
                            "teacher_checkpoint": teacher["checkpoint"],
                            "teacher_class": teacher["model_class"],
                            "teacher_path": teacher["model_path"],
                            "teacher_crf": teacher["use_crf"],
                            "student_name": student["name"],
                            "student_class": student["model_class"],
                            "student_path": student["model_path"],
                            "student_crf": student_crf,
                            "student_lr": student["lr"],
                            "data_paths": dataset,
                            "crf_combination": crf_name,
                            "alpha": alpha,
                            "beta": beta,
                            "temperature": temp,
                            "exp_name": f"KD_{teacher['name']}_{student['name']}_{crf_name}_{ab_name}_{temp_name}"
                        }
                        experiments.append(exp)
                        exp_id += 1
    
    return experiments


def load_teacher_model(checkpoint_path, model_class, model_path, label_info, use_crf, device):
    """
    Load a trained teacher model from checkpoint.
    
    Args:
        checkpoint_path: Path to checkpoint file
        model_class: Model class name
        model_path: Pretrained model path
        label_info: Label information
        use_crf: Whether teacher uses CRF
        device: Device to load on
        
    Returns:
        Loaded teacher model in eval mode
    """
    print(f"Loading teacher from: {checkpoint_path}")
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Instantiate teacher model
    if model_class == "QueryNERTeacher":
        teacher = QueryNERTeacher(
            model_name=model_path,
            label_info=label_info,
            use_crf=use_crf
        )
    else:
        raise ValueError(f"Unknown teacher class: {model_class}")
    
    # Load weights
    teacher.load_state_dict(checkpoint['model_state_dict'])
    teacher.to(device)
    teacher.eval()
    
    # Freeze teacher
    for param in teacher.parameters():
        param.requires_grad = False
    
    print(f"✓ Teacher loaded successfully")
    print(f"  Checkpoint epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f"  Checkpoint Val F1: {checkpoint.get('val_f1', 'N/A'):.4f}")
    
    return teacher


def instantiate_student(model_class, model_path, label_info, use_crf, device):
    """Instantiate a student model"""
    
    if model_class == "DistilBERTStudent":
        model = DistilBERTStudent(
            model_name=model_path,
            label_info=label_info,
            use_crf=use_crf
        )
    elif model_class == "TinyBertStudent":
        model = TinyBertStudent(
            model_name=model_path,
            label_info=label_info,
            use_crf=use_crf
        )
    elif model_class == "BiLSTMStudent":
        model = BiLSTMStudent(
            use_crf=use_crf,
            model_name_for_vocab=model_path,
            label_info=label_info
        )
    else:
        raise ValueError(f"Unknown student class: {model_class}")
    
    return model.to(device)


def evaluate_on_test(model, test_loader, device):
    """Evaluate model on test set"""
    from tqdm.auto import tqdm
    
    model.eval()
    total_loss = 0.0
    tp_acc, pred_acc, actual_acc = 0, 0, 0
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            batch_size, seq_len = input_ids.shape
            
            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = output.get("loss", torch.tensor(0.0, device=device))
            total_loss += float(loss.item())
            
            # Get predictions
            if "pred" in output:
                pred = output["pred"]
                if isinstance(pred, torch.Tensor):
                    pred_tensor = pred.to(device)
                else:
                    pred_tensor = torch.zeros((batch_size, seq_len), dtype=torch.long, device=device)
                    for i, p in enumerate(pred):
                        if isinstance(p, torch.Tensor):
                            p = p.tolist()
                        L = len(p)
                        if L > 0:
                            pred_tensor[i, :L] = torch.tensor(p, dtype=torch.long, device=device)
            else:
                pred_tensor = output["logits"].argmax(dim=-1)
            
            # Compute metrics
            mask = attention_mask.bool()
            valid = mask & (labels != -100)
            
            if valid.sum().item() > 0:
                preds_flat = pred_tensor[valid].view(-1)
                labels_flat = labels[valid].view(-1)
                
                tp = int((preds_flat == labels_flat).sum().item())
                pred_count = int(preds_flat.numel())
                actual_count = int(labels_flat.numel())
                
                tp_acc += tp
                pred_acc += pred_count
                actual_acc += actual_count
    
    avg_loss = total_loss / len(test_loader)
    precision = tp_acc / pred_acc if pred_acc > 0 else 0.0
    recall = tp_acc / actual_acc if actual_acc > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return avg_loss, 0.0, avg_loss, precision, recall, f1


def save_checkpoint(model, exp, epoch, val_f1, is_best=False):
    """Save model checkpoint"""
    checkpoint_dir = Path("/kaggle/working/checkpoints/kd")
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'val_f1': val_f1,
        'experiment': exp
    }
    
    if is_best:
        best_path = checkpoint_dir / f"{exp['exp_name']}_best.pt"
        torch.save(checkpoint, best_path)
        print(f"  ✓ Best model saved: {best_path}")


def train_single_kd_experiment(exp, label_info, device, num_epochs=15, batch_size=16, max_length=128, patience=3):
    """Train a single KD experiment"""
    
    print(f"\n{'='*80}")
    print(f"KD Experiment {exp['id']}/72: {exp['exp_name']}")
    print(f"{'='*80}")
    print(f"Teacher: {exp['teacher_name']} (CRF={exp['teacher_crf']})")
    print(f"Student: {exp['student_name']} (CRF={exp['student_crf']})")
    print(f"CRF Combination: {exp['crf_combination']}")
    print(f"Alpha (KD): {exp['alpha']}, Beta (Student): {exp['beta']}")
    print(f"Temperature: {exp['temperature']}")
    print(f"Learning Rate: {exp['student_lr']}")
    print(f"Weight Decay: 0.01")
    print(f"Early Stopping Patience: {patience}")
    print(f"{'='*80}\n")
    
    # Create dataloaders
    print("Loading data...")
    train_loader, val_loader, test_loader = create_dataloaders(
        train_path=exp['data_paths']['train'],
        val_path=exp['data_paths']['val'],
        test_path=exp['data_paths']['test'],
        model_name="bert-base-uncased",
        batch_size=batch_size,
        max_length=max_length
    )
    
    # Load teacher model
    print("\nLoading teacher model...")
    teacher = load_teacher_model(
        checkpoint_path=exp['teacher_checkpoint'],
        model_class=exp['teacher_class'],
        model_path=exp['teacher_path'],
        label_info=label_info,
        use_crf=exp['teacher_crf'],
        device=device
    )
    
    # Instantiate student model
    print(f"\nInstantiating student model: {exp['student_class']}...")
    student = instantiate_student(
        model_class=exp['student_class'],
        model_path=exp['student_path'],
        label_info=label_info,
        use_crf=exp['student_crf'],
        device=device
    )
    
    # Create optimizer
    optimizer = optim.AdamW(
        student.parameters(),
        lr=exp['student_lr'],
        weight_decay=0.01
    )
    
    # Create early stopping
    early_stopping = EarlyStopping(patience=patience, min_delta=0.001, mode='max')
    
    # Create KD trainer
    trainer = KDTrainer(
        teacher_model=teacher,
        student_model=student,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        device=device,
        alpha=exp['alpha'],
        beta=exp['beta'],
        temperature=exp['temperature']
    )
    
    # Training loop
    print(f"\nTraining for up to {num_epochs} epochs (with early stopping)...")
    history = {
        "train_loss": [], "val_loss": [],
        "train_kd": [], "val_kd": [],
        "train_stu": [], "val_stu": [],
        "train_precision": [], "train_recall": [], "train_f1": [],
        "val_precision": [], "val_recall": [], "val_f1": [],
        "test_loss": None,
        "test_precision": None,
        "test_recall": None,
        "test_f1": None,
        "stopped_epoch": None,
        "best_epoch": None
    }
    
    best_val_f1 = 0.0
    
    for epoch in range(1, num_epochs + 1):
        print(f"\n{'='*60}")
        print(f"EPOCH {epoch}/{num_epochs}")
        print(f"{'='*60}")
        
        # Train and validate
        train_loss, train_kd, train_stu, train_prec, train_rec, train_f1 = trainer.train_epoch()
        val_loss, val_kd, val_stu, val_prec, val_rec, val_f1 = trainer.validate()
        
        print(f"\nTrain Loss: {train_loss:.4f} (KD: {train_kd:.4f}, Student: {train_stu:.4f})")
        print(f"Val Loss:   {val_loss:.4f} (KD: {val_kd:.4f}, Student: {val_stu:.4f})")
        print(f"\nTrain Metrics -> P: {train_prec:.4f}, R: {train_rec:.4f}, F1: {train_f1:.4f}")
        print(f"Val Metrics   -> P: {val_prec:.4f}, R: {val_rec:.4f}, F1: {val_f1:.4f}")
        
        # Store history
        history["train_loss"].append(train_loss)
        history["train_kd"].append(train_kd)
        history["train_stu"].append(train_stu)
        history["val_loss"].append(val_loss)
        history["val_kd"].append(val_kd)
        history["val_stu"].append(val_stu)
        history["train_precision"].append(train_prec)
        history["train_recall"].append(train_rec)
        history["train_f1"].append(train_f1)
        history["val_precision"].append(val_prec)
        history["val_recall"].append(val_rec)
        history["val_f1"].append(val_f1)
        
        # Check if best model
        is_best = val_f1 > best_val_f1
        if is_best:
            best_val_f1 = val_f1
            print(f"  ✓ New best Val F1: {best_val_f1:.4f}")
            save_checkpoint(student, exp, epoch, val_f1, is_best=True)
        
        # Check early stopping
        if early_stopping(val_f1, epoch):
            history["stopped_epoch"] = epoch
            history["best_epoch"] = early_stopping.best_epoch
            print(f"\n{'='*60}")
            print(f"Training stopped early at epoch {epoch}")
            print(f"Best validation F1: {best_val_f1:.4f} at epoch {early_stopping.best_epoch}")
            print(f"{'='*60}")
            break
    else:
        history["stopped_epoch"] = num_epochs
        history["best_epoch"] = history["val_f1"].index(max(history["val_f1"])) + 1
        print(f"\n{'='*60}")
        print(f"Training completed all {num_epochs} epochs")
        print(f"Best validation F1: {best_val_f1:.4f} at epoch {history['best_epoch']}")
        print(f"{'='*60}")
    
    # Load best model for test evaluation
    print(f"\n{'='*60}")
    print("LOADING BEST MODEL FOR TEST EVALUATION")
    print(f"{'='*60}")
    checkpoint_path = Path("/kaggle/working/checkpoints/kd") / f"{exp['exp_name']}_best.pt"
    checkpoint = torch.load(checkpoint_path, map_location=device)
    student.load_state_dict(checkpoint['model_state_dict'])
    student.eval()
    print(f"✓ Loaded best model from epoch {checkpoint['epoch']} (Val F1: {checkpoint['val_f1']:.4f})")
    
    # Evaluate on test set
    print(f"\n{'='*60}")
    print("EVALUATING ON TEST SET")
    print(f"{'='*60}")
    test_loss, test_kd, test_stu, test_prec, test_rec, test_f1 = evaluate_on_test(
        model=student,
        test_loader=test_loader,
        device=device
    )
    
    print(f"\nTest Results:")
    print(f"  Loss: {test_loss:.4f}")
    print(f"  Precision: {test_prec:.4f}")
    print(f"  Recall: {test_rec:.4f}")
    print(f"  F1: {test_f1:.4f}")
    
    # Add test metrics to history
    history["test_loss"] = test_loss
    history["test_precision"] = test_prec
    history["test_recall"] = test_rec
    history["test_f1"] = test_f1
    
    # Save results
    save_experiment_results(exp, history)
    
    # Clear memory
    del teacher, student, trainer, optimizer, checkpoint
    torch.cuda.empty_cache()
    
    return history


def save_experiment_results(exp, history):
    """Save experiment results"""
    base_dir = Path("/kaggle/working/results/kd")
    json_dir = base_dir / "json"
    
    json_dir.mkdir(parents=True, exist_ok=True)
    
    json_path = json_dir / f"{exp['exp_name']}.json"
    
    result_data = {
        "experiment": exp,
        "history": history,
        "timestamp": datetime.now().isoformat()
    }
    
    with open(json_path, "w") as f:
        json.dump(result_data, f, indent=4)
    
    print(f"\n✓ Results saved to: {json_path}")


def run_all_kd_experiments(
    device="cuda",
    num_epochs=15,
    batch_size=16,
    max_length=128,
    patience=3,
    start_from=1,
    end_at=72
):
    """Run all 72 KD experiments"""
    
    print("\n" + "="*80)
    print("KNOWLEDGE DISTILLATION TRAINING: 72 EXPERIMENTS")
    print("="*80)
    print(f"Device: {device}")
    print(f"Max Epochs: {num_epochs}")
    print(f"Early Stopping Patience: {patience}")
    print(f"Batch Size: {batch_size}")
    print(f"Max Length: {max_length}")
    print(f"Weight Decay: 0.01")
    print(f"Dropout: 0.3 (in model definitions)")
    print(f"Running experiments {start_from} to {end_at}")
    print("="*80 + "\n")
    
    # Load label info
    print("Loading label information...")
    label_info = load_label_info("bltlab/queryner-augmented-data-bert-base-uncased")
    print(f"Number of labels: {label_info['num_labels']}")
    
    # Generate all KD experiment configs
    experiments = create_kd_experiment_config()
    
    print(f"\nTotal KD experiments generated: {len(experiments)}")
    
    # Filter experiments
    experiments = [exp for exp in experiments if start_from <= exp['id'] <= end_at]
    
    print(f"Experiments to run: {len(experiments)}\n")
    
    # Track results
    all_results = []
    failed_experiments = []
    
    # Run each experiment
    for i, exp in enumerate(experiments, 1):
        try:
            history = train_single_kd_experiment(
                exp=exp,
                label_info=label_info,
                device=device,
                num_epochs=num_epochs,
                batch_size=batch_size,
                max_length=max_length,
                patience=patience
            )
            
            # Store summary
            final_metrics = {
                "exp_name": exp['exp_name'],
                "teacher": exp['teacher_name'],
                "student": exp['student_name'],
                "crf_combo": exp['crf_combination'],
                "alpha": exp['alpha'],
                "beta": exp['beta'],
                "temperature": exp['temperature'],
                "val_f1": history['val_f1'][-1],
                "best_val_f1": max(history['val_f1']),
                "test_f1": history['test_f1'],
                "best_epoch": history['best_epoch'],
                "stopped_epoch": history['stopped_epoch'],
                "early_stopped": history['stopped_epoch'] < num_epochs
            }
            all_results.append(final_metrics)
            
            print(f"\n✓ Experiment {exp['id']} completed successfully!")
            print(f"Best Val F1: {final_metrics['best_val_f1']:.4f} (epoch {final_metrics['best_epoch']})")
            print(f"Test F1: {final_metrics['test_f1']:.4f}")
            print(f"Stopped at epoch: {final_metrics['stopped_epoch']}")
            
        except Exception as e:
            print(f"\n✗ Experiment {exp['id']} FAILED!")
            print(f"Error: {str(e)}\n")
            import traceback
            traceback.print_exc()
            failed_experiments.append({
                "exp_id": exp['id'],
                "exp_name": exp['exp_name'],
                "error": str(e)
            })
            continue
    
    # Save summary
    save_kd_summary(all_results, failed_experiments)
    
    print("\n" + "="*80)
    print("ALL KD EXPERIMENTS COMPLETED")
    print("="*80)
    print(f"Successful: {len(all_results)}")
    print(f"Failed: {len(failed_experiments)}")
    print("="*80 + "\n")


def save_kd_summary(all_results, failed_experiments):
    """Save summary of all KD experiments"""
    
    summary_dir = Path("/kaggle/working/results/kd")
    summary_dir.mkdir(parents=True, exist_ok=True)
    
    summary_path = summary_dir / "summary.json"
    with open(summary_path, "w") as f:
        json.dump({
            "successful_experiments": all_results,
            "failed_experiments": failed_experiments,
            "timestamp": datetime.now().isoformat()
        }, f, indent=4)
    
    print(f"\n✓ Summary saved to: {summary_path}")
    
    if all_results:
        print("\n" + "="*80)
        print("KD TRAINING STATISTICS")
        print("="*80)
        
        # Early stopping stats
        early_stopped = [r for r in all_results if r['early_stopped']]
        print(f"\nEarly Stopping Statistics:")
        print(f"  Experiments stopped early: {len(early_stopped)}/{len(all_results)}")
        if early_stopped:
            avg_stopped = sum(r['stopped_epoch'] for r in early_stopped) / len(early_stopped)
            print(f"  Average stopping epoch: {avg_stopped:.1f}")
        
        # Top models by validation F1
        print("\n" + "="*80)
        print("Top 10 KD Models by Best Val F1:")
        print("="*80)
        sorted_results = sorted(all_results, key=lambda x: x['best_val_f1'], reverse=True)
        for i, result in enumerate(sorted_results[:10], 1):
            print(f"{i:2d}. {result['exp_name']:60s}")
            print(f"    Val F1: {result['best_val_f1']:.4f} | Test F1: {result['test_f1']:.4f}")
            print(f"    Teacher: {result['teacher']:15s} | Student: {result['student']:10s}")
            print(f"    α={result['alpha']}, β={result['beta']}, T={result['temperature']}, CRF={result['crf_combo']}")
        
        # Top models by test F1
        print("\n" + "="*80)
        print("Top 10 KD Models by Test F1:")
        print("="*80)
        sorted_by_test = sorted(all_results, key=lambda x: x['test_f1'], reverse=True)
        for i, result in enumerate(sorted_by_test[:10], 1):
            print(f"{i:2d}. {result['exp_name']:60s}")
            print(f"    Test F1: {result['test_f1']:.4f} | Val F1: {result['best_val_f1']:.4f}")
            print(f"    Teacher: {result['teacher']:15s} | Student: {result['student']:10s}")
        
        # Best per student
        print("\n" + "="*80)
        print("Best Configuration Per Student:")
        print("="*80)
        for student_name in ['distilbert', 'tinybert', 'bilstm']:
            student_results = [r for r in all_results if r['student'] == student_name]
            if student_results:
                best = max(student_results, key=lambda x: x['test_f1'])
                print(f"\n{student_name.upper()}:")
                print(f"  Best Test F1: {best['test_f1']:.4f}")
                print(f"  Configuration: {best['exp_name']}")

In [ ]:
# Example usage
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    # Run all KD experiments
    # You can run in batches:
    # run_all_kd_experiments(device=device, start_from=1, end_at=18)   # First 18
    # run_all_kd_experiments(device=device, start_from=19, end_at=36)  # Next 18
    # run_all_kd_experiments(device=device, start_from=37, end_at=54)  # Next 18
    # run_all_kd_experiments(device=device, start_from=55, end_at=72)  # Last 18
    
    run_all_kd_experiments(
        device=device,
        num_epochs=30,
        batch_size=32,
        max_length=128,
        patience=3,
        start_from=1,
        end_at=12
    )

## Baseline Model

In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
# label_info = load_label_info("bltlab/queryner-augmented-data-bert-base-uncased")
# teacher_for_bilstm = QueryNERTeacher(
#     model_name="bert-base-uncased",
#     label_info=label_info,
#     use_crf=False
# )
# teacher_for_bilstm.to(device)

# ckpt_path = r"/kaggle/input/teacher-processed-2e-5-nocrf-best/pytorch/default/3/teacher_processed_2e-5_nocrf_best_token_mean.pt"
# state = torch.load(ckpt_path, map_location=device)  # state is likely a dict of tensors

# if all(isinstance(v, torch.Tensor) for v in state.values()):
#     teacher_for_bilstm.load_state_dict(state)
# else:
#     if "model_state_dict" in state:
#         teacher_for_bilstm.load_state_dict(state["model_state_dict"])
#     elif "state_dict" in state:
#         teacher_for_bilstm.load_state_dict(state["state_dict"])
#     else:
#         for k, v in state.items():
#             if isinstance(v, dict) and any(isinstance(t, torch.Tensor) for t in v.values()):
#                 teacher_for_bilstm.load_state_dict(v)
#                 break

In [ ]:
# """
# Comprehensive Baseline Training Script for NER Models
# Trains 4 models × 2 datasets × 3 learning rates × 2 CRF settings = 48 experiments
# With Early Stopping and Best Model Checkpointing
# """

# import json
# import os
# import torch
# import torch.optim as optim
# from datetime import datetime
# from pathlib import Path

# # Import your existing modules (assumed to be available)
# # from your_module import (
# #     NERDataset, create_dataloaders, load_label_info,
# #     QueryNERTeacher, DistilBERTStudent, TinyBertStudent, BiLSTMStudent,
# #     KDTrainer
# # )


# class EarlyStopping:
#     """Early stopping to stop training when validation metric stops improving."""
    
#     def __init__(self, patience=3, min_delta=0.001, mode='max'):
#         """
#         Args:
#             patience: Number of epochs to wait for improvement
#             min_delta: Minimum change to qualify as improvement
#             mode: 'max' for metrics like F1 (higher is better), 'min' for loss
#         """
#         self.patience = patience
#         self.min_delta = min_delta
#         self.mode = mode
#         self.counter = 0
#         self.best_score = None
#         self.early_stop = False
#         self.best_epoch = 0
        
#     def __call__(self, val_metric, epoch):
#         """
#         Check if training should stop.
        
#         Args:
#             val_metric: Current validation metric value
#             epoch: Current epoch number
            
#         Returns:
#             bool: True if should stop, False otherwise
#         """
#         score = val_metric
        
#         if self.best_score is None:
#             self.best_score = score
#             self.best_epoch = epoch
#             return False
        
#         # Check for improvement based on mode
#         if self.mode == 'max':
#             improved = score > self.best_score + self.min_delta
#         else:  # mode == 'min'
#             improved = score < self.best_score - self.min_delta
        
#         if improved:
#             self.best_score = score
#             self.best_epoch = epoch
#             self.counter = 0
#         else:
#             self.counter += 1
#             print(f"  EarlyStopping counter: {self.counter}/{self.patience}")
#             if self.counter >= self.patience:
#                 self.early_stop = True
#                 print(f"  ✓ Early stopping triggered! Best epoch was {self.best_epoch}")
        
#         return self.early_stop


# def create_experiment_config():
#     """Generate all 48 experiment configurations"""
    
#     # Configuration space
#     models = [
#         # ("teacher", "QueryNERTeacher", "bert-base-uncased"),
#         # ("distilbert", "DistilBERTStudent", "distilbert-base-uncased"),
#         # ("tinybert", "TinyBertStudent", "huawei-noah/TinyBERT_General_4L_312D"),
#         ("bilstm", "BiLSTMStudent", "bert-base-uncased")
#     ]
    
#     datasets = [
#         ("processed", {
#             "train": r"/kaggle/input/queryner-dataset/data/processed/train.json",
#             "val": r"/kaggle/input/queryner-dataset/data/processed/validation.json",
#             "test": r"/kaggle/input/queryner-dataset/data/processed/test.json"
#         })
#         # ,("raw", {
#         #     "train": r"/kaggle/input/queryner-dataset/data/raw/train.json",
#         #     "val": r"/kaggle/input/queryner-dataset/data/raw/validation.json",
#         #     "test": r"/kaggle/input/queryner-dataset/data/raw/test.json"
#         # })
#     ]
    
#     learning_rates = [
#         # (1e-5, "1e-5"),   # Very conservative
#         # (2e-5, "2e-5"),   # Conservative, stable
#         # (5e-5, "5e-5")    # Balanced speed/stability

#         # for bilstm model trial
#         # (2e-4, "2e-4"),
#         (5e-4, "5e-4"),
#         (1e-3, "1e-3"),
#         (2e-3, "2e-3"),
#         (3e-3, "3e-3"),
#         # (1e-2, "1e-2")
#     ]
    
#     crf_settings = [
#         # (True, "crf"),
#         (False, "nocrf")
#     ]
    
#     # Generate all combinations
#     experiments = []
#     exp_id = 1
    
#     for model_name, model_class, model_path in models:
#         for data_name, data_paths in datasets:
#             for lr_value, lr_name in learning_rates:
#                 for use_crf, crf_name in crf_settings:
#                     exp = {
#                         "id": exp_id,
#                         "model_name": model_name,
#                         "model_class": model_class,
#                         "model_path": model_path,
#                         "data_name": data_name,
#                         "data_paths": data_paths,
#                         "learning_rate": lr_value,
#                         "lr_name": lr_name,
#                         "use_crf": use_crf,
#                         "crf_name": crf_name,
#                         "exp_name": f"{model_name}_{data_name}_{lr_name}_{crf_name}"
#                     }
#                     experiments.append(exp)
#                     exp_id += 1
    
#     return experiments


# def evaluate_on_test(model, test_loader, device):
#     """
#     Evaluate model on test set.
    
#     Args:
#         model: Trained model (should be in eval mode)
#         test_loader: Test DataLoader
#         device: Device to use
        
#     Returns:
#         test_loss, test_kd, test_stu, test_precision, test_recall, test_f1
#     """
#     from tqdm.auto import tqdm
    
#     model.eval()
#     total_loss = 0.0
#     tp_acc, pred_acc, actual_acc = 0, 0, 0
    
#     with torch.no_grad():
#         for batch in tqdm(test_loader, desc="Testing"):
#             input_ids = batch["input_ids"].to(device)
#             attention_mask = batch["attention_mask"].to(device)
#             labels = batch["labels"].to(device)
            
#             batch_size, seq_len = input_ids.shape
            
#             # Get model outputs
#             output = model(
#                 input_ids=input_ids,
#                 attention_mask=attention_mask,
#                 labels=labels
#             )
            
#             # Get loss
#             loss = output.get("loss", torch.tensor(0.0, device=device))
#             total_loss += float(loss.item())
            
#             # Get predictions
#             if "pred" in output:
#                 pred = output["pred"]
#                 if isinstance(pred, torch.Tensor):
#                     pred_tensor = pred.to(device)
#                 else:
#                     # Handle CRF list output
#                     pred_tensor = torch.zeros((batch_size, seq_len), dtype=torch.long, device=device)
#                     for i, p in enumerate(pred):
#                         if isinstance(p, torch.Tensor):
#                             p = p.tolist()
#                         L = len(p)
#                         if L > 0:
#                             pred_tensor[i, :L] = torch.tensor(p, dtype=torch.long, device=device)
#             else:
#                 pred_tensor = output["logits"].argmax(dim=-1)
            
#             # Compute metrics
#             mask = attention_mask.bool()
#             valid = mask & (labels != -100)
            
#             if valid.sum().item() > 0:
#                 preds_flat = pred_tensor[valid].view(-1)
#                 labels_flat = labels[valid].view(-1)
                
#                 # Compute TP, predicted, actual counts
#                 tp = int((preds_flat == labels_flat).sum().item())
#                 pred_count = int(preds_flat.numel())
#                 actual_count = int(labels_flat.numel())
                
#                 tp_acc += tp
#                 pred_acc += pred_count
#                 actual_acc += actual_count
    
#     # Compute averages
#     avg_loss = total_loss / len(test_loader)
    
#     # Compute final metrics
#     precision = tp_acc / pred_acc if pred_acc > 0 else 0.0
#     recall = tp_acc / actual_acc if actual_acc > 0 else 0.0
#     if precision + recall > 0:
#         f1 = 2 * precision * recall / (precision + recall)
#     else:
#         f1 = 0.0
    
#     return avg_loss, 0.0, avg_loss, precision, recall, f1


# def instantiate_model(model_class, model_path, label_info, use_crf, device):
#     """Instantiate the correct model based on class name"""
    
#     if model_class == "QueryNERTeacher":
#         model = QueryNERTeacher(
#             model_name=model_path,
#             label_info=label_info,
#             use_crf=use_crf
#         )
#     elif model_class == "DistilBERTStudent":
#         model = DistilBERTStudent(
#             model_name=model_path,
#             label_info=label_info,
#             use_crf=use_crf
#         )
#     elif model_class == "TinyBertStudent":
#         model = TinyBertStudent(
#             model_name=model_path,
#             label_info=label_info,
#             use_crf=use_crf
#         )
#     elif model_class == "BiLSTMStudent":
#         model = BiLSTMStudent(
#             # num_labels=label_info["num_labels"],
#             use_crf=use_crf,
#             model_name_for_vocab=model_path,
#             label_info=label_info,
#             teacher_model=teacher_for_bilstm
#         )
#     else:
#         raise ValueError(f"Unknown model class: {model_class}")
    
#     return model.to(device)


# def save_checkpoint(model, exp, epoch, val_f1, is_best=False):
#     """
#     Save model checkpoint.
    
#     Args:
#         model: Model to save
#         exp: Experiment configuration dict
#         epoch: Current epoch
#         val_f1: Validation F1 score
#         is_best: Whether this is the best model so far
#     """
#     # Create checkpoints directory
#     checkpoint_dir = Path("/kaggle/working/checkpoints/baseline")
#     checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
#     # Prepare checkpoint data
#     checkpoint = {
#         'epoch': epoch,
#         'model_state_dict': model.state_dict(),
#         'val_f1': val_f1,
#         'experiment': exp
#     }
    
#     # Save best model
#     if is_best:
#         best_path = checkpoint_dir / f"{exp['exp_name']}_best.pt"
#         torch.save(checkpoint, best_path)
#         print(f"  ✓ Best model saved: {best_path}")


# def train_single_experiment(exp, label_info, device, num_epochs=15, batch_size=16, max_length=128, patience=3):
#     """Train a single experiment configuration with early stopping"""
    
#     print(f"\n{'='*80}")
#     print(f"Experiment {exp['id']}/6: {exp['exp_name']}")
#     print(f"{'='*80}")
#     print(f"Model: {exp['model_name']}")
#     print(f"Dataset: {exp['data_name']}")
#     print(f"Learning Rate: {exp['lr_name']}")
#     print(f"CRF: {exp['use_crf']}")
#     print(f"Weight Decay: 0.01")
#     print(f"Early Stopping Patience: {patience}")
#     print(f"{'='*80}\n")
    
#     # Create dataloaders
#     print("Loading data...")
#     train_loader, val_loader, test_loader = create_dataloaders(
#         train_path=exp['data_paths']['train'],
#         val_path=exp['data_paths']['val'],
#         test_path=exp['data_paths']['test'],
#         model_name="bert-base-uncased",  # tokenizer
#         batch_size=batch_size,
#         max_length=max_length
#     )
    
#     # Instantiate model
#     print(f"Instantiating model: {exp['model_class']}...")
#     model = instantiate_model(
#         model_class=exp['model_class'],
#         model_path=exp['model_path'],
#         label_info=label_info,
#         use_crf=exp['use_crf'],
#         device=device
#     )
    
#     # Create optimizer with weight decay
#     optimizer = optim.AdamW(
#         model.parameters(), 
#         lr=exp['learning_rate'], 
#         weight_decay=0.01  # L2 regularization
#     )
    
#     # Create early stopping
#     early_stopping = EarlyStopping(patience=patience, min_delta=0.001, mode='max')
    
#     # Create trainer (using KD trainer with alpha=0, beta=1 for baseline)
#     trainer = KDTrainer(
#         teacher_model=None,  # No teacher for baseline
#         student_model=model,
#         train_loader=train_loader,
#         val_loader=val_loader,
#         optimizer=optimizer,
#         device=device,
#         alpha=0.0,  # No KD loss
#         beta=1.0,   # Only student loss
#         temperature=2.0  # Not used when alpha=0
#     )
    
#     # Training loop with early stopping
#     print(f"Training for up to {num_epochs} epochs (with early stopping)...")
#     history = {
#         "train_loss": [], "val_loss": [],
#         "train_kd": [], "val_kd": [],
#         "train_stu": [], "val_stu": [],
#         "train_precision": [], "train_recall": [], "train_f1": [],
#         "val_precision": [], "val_recall": [], "val_f1": [],
#         "test_loss": None,
#         "test_precision": None,
#         "test_recall": None,
#         "test_f1": None,
#         "stopped_epoch": None,
#         "best_epoch": None
#     }
    
#     best_val_f1 = 0.0
    
#     for epoch in range(1, num_epochs + 1):
#         print(f"\n{'='*60}")
#         print(f"EPOCH {epoch}/{num_epochs}")
#         print(f"{'='*60}")
        
#         # Train and validate
#         train_loss, train_kd, train_stu, train_prec, train_rec, train_f1 = trainer.train_epoch()
#         val_loss, val_kd, val_stu, val_prec, val_rec, val_f1 = trainer.validate()
        
#         print(f"\nTrain Loss: {train_loss:.4f} (KD: {train_kd:.4f}, Student: {train_stu:.4f})")
#         print(f"Val Loss:   {val_loss:.4f} (KD: {val_kd:.4f}, Student: {val_stu:.4f})")
#         print(f"\nTrain Metrics -> P: {train_prec:.4f}, R: {train_rec:.4f}, F1: {train_f1:.4f}")
#         print(f"Val Metrics   -> P: {val_prec:.4f}, R: {val_rec:.4f}, F1: {val_f1:.4f}")
        
#         # Store history
#         history["train_loss"].append(train_loss)
#         history["train_kd"].append(train_kd)
#         history["train_stu"].append(train_stu)
#         history["val_loss"].append(val_loss)
#         history["val_kd"].append(val_kd)
#         history["val_stu"].append(val_stu)
#         history["train_precision"].append(train_prec)
#         history["train_recall"].append(train_rec)
#         history["train_f1"].append(train_f1)
#         history["val_precision"].append(val_prec)
#         history["val_recall"].append(val_rec)
#         history["val_f1"].append(val_f1)
        
#         # Check if this is the best model
#         is_best = val_f1 > best_val_f1
#         if is_best:
#             best_val_f1 = val_f1
#             print(f"  ✓ New best Val F1: {best_val_f1:.4f}")
#             save_checkpoint(model, exp, epoch, val_f1, is_best=True)
        
#         # Check early stopping
#         if early_stopping(val_f1, epoch):
#             history["stopped_epoch"] = epoch
#             history["best_epoch"] = early_stopping.best_epoch
#             print(f"\n{'='*60}")
#             print(f"Training stopped early at epoch {epoch}")
#             print(f"Best validation F1: {best_val_f1:.4f} at epoch {early_stopping.best_epoch}")
#             print(f"{'='*60}")
#             break
#     else:
#         # Training completed without early stopping
#         history["stopped_epoch"] = num_epochs
#         history["best_epoch"] = history["val_f1"].index(max(history["val_f1"])) + 1
#         print(f"\n{'='*60}")
#         print(f"Training completed all {num_epochs} epochs")
#         print(f"Best validation F1: {best_val_f1:.4f} at epoch {history['best_epoch']}")
#         print(f"{'='*60}")
    
#     # Load best model checkpoint for test evaluation
#     print(f"\n{'='*60}")
#     print("LOADING BEST MODEL FOR TEST EVALUATION")
#     print(f"{'='*60}")
#     checkpoint_path = Path("/kaggle/working/checkpoints/baseline") / f"{exp['exp_name']}_best.pt"
#     checkpoint = torch.load(checkpoint_path, map_location=device)
#     model.load_state_dict(checkpoint['model_state_dict'])
#     model.eval()
#     print(f"✓ Loaded best model from epoch {checkpoint['epoch']} (Val F1: {checkpoint['val_f1']:.4f})")
    
#     # Evaluate on test set
#     print(f"\n{'='*60}")
#     print("EVALUATING ON TEST SET")
#     print(f"{'='*60}")
#     test_loss, test_kd, test_stu, test_prec, test_rec, test_f1 = evaluate_on_test(
#         model=model,
#         test_loader=test_loader,
#         device=device
#     )
    
#     print(f"\nTest Results:")
#     print(f"  Loss: {test_loss:.4f}")
#     print(f"  Precision: {test_prec:.4f}")
#     print(f"  Recall: {test_rec:.4f}")
#     print(f"  F1: {test_f1:.4f}")
    
#     # Add test metrics to history
#     history["test_loss"] = test_loss
#     history["test_precision"] = test_prec
#     history["test_recall"] = test_rec
#     history["test_f1"] = test_f1
    
#     # Save final results (now includes test metrics)
#     save_experiment_results(exp, history)
    
#     # Clear memory
#     del model, trainer, optimizer, checkpoint
#     torch.cuda.empty_cache()
    
#     return history


# def save_experiment_results(exp, history):
#     """Save experiment results in organized structure"""
    
#     # Create directory structure
#     base_dir = Path("/kaggle/working/results/baseline")
#     json_dir = base_dir / "json"
#     img_dir = base_dir / "img"
    
#     json_dir.mkdir(parents=True, exist_ok=True)
#     img_dir.mkdir(parents=True, exist_ok=True)
    
#     # Save history as JSON
#     json_path = json_dir / f"{exp['exp_name']}.json"
    
#     result_data = {
#         "experiment": exp,
#         "history": history,
#         "timestamp": datetime.now().isoformat()
#     }
    
#     with open(json_path, "w") as f:
#         json.dump(result_data, f, indent=4)
    
#     print(f"\n✓ Results saved to: {json_path}")


# def run_all_baselines(
#     device="cuda",
#     num_epochs=15,
#     batch_size=16,
#     max_length=128,
#     patience=3,
#     start_from=1,
#     end_at=6
# ):
#     """Run all 48 baseline experiments with early stopping"""
    
#     print("\n" + "="*80)
#     print("BASELINE TRAINING: 48 EXPERIMENTS")
#     print("="*80)
#     print(f"Device: {device}")
#     print(f"Max Epochs: {num_epochs}")
#     print(f"Early Stopping Patience: {patience}")
#     print(f"Batch Size: {batch_size}")
#     print(f"Max Length: {max_length}")
#     print(f"Weight Decay: 0.01")
#     print(f"Dropout: 0.3 (in model definitions)")
#     print(f"Running experiments {start_from} to {end_at}")
#     print("="*80 + "\n")
    
#     # Load label info (use teacher model config)
#     print("Loading label information...")
#     label_info = load_label_info("bltlab/queryner-augmented-data-bert-base-uncased")
#     print(f"Number of labels: {label_info['num_labels']}")
    
#     # Generate all experiment configs
#     experiments = create_experiment_config()
    
#     # Filter experiments based on start_from and end_at
#     experiments = [exp for exp in experiments if start_from <= exp['id'] <= end_at]
    
#     print(f"\nTotal experiments to run: {len(experiments)}\n")
    
#     # Track results
#     all_results = []
#     failed_experiments = []
    
#     # Run each experiment
#     for i, exp in enumerate(experiments, 1):
#         try:
#             history = train_single_experiment(
#                 exp=exp,
#                 label_info=label_info,
#                 device=device,
#                 num_epochs=num_epochs,
#                 batch_size=batch_size,
#                 max_length=max_length,
#                 patience=patience
#             )
            
#             # Store summary
#             final_metrics = {
#                 "exp_name": exp['exp_name'],
#                 "val_f1": history['val_f1'][-1],
#                 "val_precision": history['val_precision'][-1],
#                 "val_recall": history['val_recall'][-1],
#                 "best_val_f1": max(history['val_f1']),
#                 "test_f1": history['test_f1'],
#                 "test_precision": history['test_precision'],
#                 "test_recall": history['test_recall'],
#                 "best_epoch": history['best_epoch'],
#                 "stopped_epoch": history['stopped_epoch'],
#                 "early_stopped": history['stopped_epoch'] < num_epochs
#             }
#             all_results.append(final_metrics)
            
#             print(f"\n✓ Experiment {exp['id']} completed successfully!")
#             print(f"Best Val F1: {final_metrics['best_val_f1']:.4f} (epoch {final_metrics['best_epoch']})")
#             print(f"Test F1: {final_metrics['test_f1']:.4f}")
#             print(f"Stopped at epoch: {final_metrics['stopped_epoch']}")
            
#         except Exception as e:
#             print(f"\n✗ Experiment {exp['id']} FAILED!")
#             print(f"Error: {str(e)}\n")
#             failed_experiments.append({
#                 "exp_id": exp['id'],
#                 "exp_name": exp['exp_name'],
#                 "error": str(e)
#             })
#             continue
    
#     # Save summary
#     save_summary(all_results, failed_experiments)
    
#     print("\n" + "="*80)
#     print("ALL EXPERIMENTS COMPLETED")
#     print("="*80)
#     print(f"Successful: {len(all_results)}")
#     print(f"Failed: {len(failed_experiments)}")
#     print("="*80 + "\n")


# def save_summary(all_results, failed_experiments):
#     """Save summary of all experiments"""
    
#     summary_dir = Path("/kaggle/working/results/baseline")
#     summary_dir.mkdir(parents=True, exist_ok=True)
    
#     # Save results summary
#     summary_path = summary_dir / "summary.json"
#     with open(summary_path, "w") as f:
#         json.dump({
#             "successful_experiments": all_results,
#             "failed_experiments": failed_experiments,
#             "timestamp": datetime.now().isoformat()
#         }, f, indent=4)
    
#     print(f"\n✓ Summary saved to: {summary_path}")
    
#     # Print statistics
#     if all_results:
#         print("\n" + "="*80)
#         print("TRAINING STATISTICS")
#         print("="*80)
        
#         # Early stopping stats
#         early_stopped = [r for r in all_results if r['early_stopped']]
#         print(f"\nEarly Stopping Statistics:")
#         print(f"  Experiments stopped early: {len(early_stopped)}/{len(all_results)}")
#         if early_stopped:
#             avg_stopped = sum(r['stopped_epoch'] for r in early_stopped) / len(early_stopped)
#             print(f"  Average stopping epoch: {avg_stopped:.1f}")
        
#         # Top performing models
#         print("\n" + "="*80)
#         print("Top 10 Models by Best Val F1:")
#         print("="*80)
#         sorted_results = sorted(all_results, key=lambda x: x['best_val_f1'], reverse=True)
#         for i, result in enumerate(sorted_results[:10], 1):
#             early_marker = "⚡" if result['early_stopped'] else "  "
#             print(f"{i:2d}. {early_marker} {result['exp_name']:45s} | "
#                   f"Val F1: {result['best_val_f1']:.4f} | "
#                   f"Test F1: {result['test_f1']:.4f} | "
#                   f"Epoch: {result['best_epoch']:2d}/{result['stopped_epoch']:2d}")
        
#         # Best test F1 models
#         print("\n" + "="*80)
#         print("Top 10 Models by Test F1:")
#         print("="*80)
#         sorted_by_test = sorted(all_results, key=lambda x: x['test_f1'], reverse=True)
#         for i, result in enumerate(sorted_by_test[:10], 1):
#             early_marker = "⚡" if result['early_stopped'] else "  "
#             print(f"{i:2d}. {early_marker} {result['exp_name']:45s} | "
#                   f"Test F1: {result['test_f1']:.4f} | "
#                   f"Val F1: {result['best_val_f1']:.4f}")


# def load_best_model(exp_name, model_class, model_path, label_info, device):
#     """
#     Load the best saved model for an experiment.
    
#     Args:
#         exp_name: Experiment name (e.g., "distilbert_processed_2e-5_crf")
#         model_class: Model class name
#         model_path: Path to pretrained model
#         label_info: Label information dict
#         device: Device to load model on
        
#     Returns:
#         Loaded model with best weights
#     """
#     # Find checkpoint file
#     checkpoint_dir = Path("/kaggle/working/checkpoints/baseline")
#     checkpoint_path = checkpoint_dir / f"{exp_name}_best.pt"
    
#     if not checkpoint_path.exists():
#         raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    
#     # Load checkpoint
#     checkpoint = torch.load(checkpoint_path, map_location=device)
    
#     # Get use_crf from experiment config
#     use_crf = checkpoint['experiment']['use_crf']
    
#     # Instantiate model
#     model = instantiate_model(model_class, model_path, label_info, use_crf, device)
    
#     # Load weights
#     model.load_state_dict(checkpoint['model_state_dict'])
#     model.eval()
    
#     print(f"✓ Loaded best model from epoch {checkpoint['epoch']}")
#     print(f"  Val F1: {checkpoint['val_f1']:.4f}")
    
#     return model


# # Example usage
# if __name__ == "__main__":
#     # Check device
#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     print(f"Using device: {device}")
    
#     # Run all baselines with early stopping
#     # You can also run in batches:
#     # run_all_baselines(device=device, start_from=1, end_at=12)   # First 12
#     # run_all_baselines(device=device, start_from=13, end_at=24)  # Next 12
#     # run_all_baselines(device=device, start_from=25, end_at=36)  # Next 12
#     # run_all_baselines(device=device, start_from=37, end_at=48)  # Last 12
    
#     run_all_baselines(
#         device=device,
#         num_epochs=15,      # Max epochs (will stop early if no improvement)
#         batch_size=32,
#         max_length=128,
#         patience=3,         # Stop if no improvement for 3 epochs
#         start_from=1,
#         end_at=4
#     )
    
#     # Example: Load best model after training
#     # label_info = load_label_info("bltlab/queryner-augmented-data-bert-base-uncased")
#     # best_model = load_best_model(
#     #     exp_name="distilbert_processed_2e-5_crf",
#     #     model_class="DistilBERTStudent",
#     #     model_path="distilbert-base-uncased",
#     #     label_info=label_info,
#     #     device=device
#     # )